# Create External Locations

Register ADLS Gen2 containers as Unity Catalog **External Locations** secured by a Storage Credential.

> **Prerequisites**
> - The Storage Credential `dbstoragefinancestoragetoken` must already exist in Unity Catalog (created via the Databricks Account Console or Terraform).
> - Run `0.config` first — all path and credential values come from Widgets so nothing is hardcoded here.
> - Widget values are validated by `0.config` to contain only alphanumeric, underscore, and hyphen characters before being interpolated into any SQL statement.

> **Note on `CATALOG_ROOT`:** The `finance-{ENV}` container is registered as the catalog's `MANAGED LOCATION` in `2.create_catalog_schema`. Do not also register it as an external location to avoid a path-ownership conflict in Unity Catalog.


In [0]:
%run ./0.config

In [0]:
# ── Step 1: Register External Locations ──────────────────────────────────────
# STORAGE_ACCOUNT and STORAGE_CREDENTIAL are pre-validated in 0.config
# (allow-list regex: ^[A-Za-z0-9_\-]+$) before being interpolated here.
# CATALOG_ROOT (finance-{ENV}) is intentionally excluded — it is registered
# as the catalog's MANAGED LOCATION in notebook 2, not as an External Location.

for container in ["bronze", "silver", "gold"]:
    loc_name = f"{STORAGE_ACCOUNT}_{container}"
    loc_url  = abfs(container)
    spark.sql(f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS {loc_name}
    URL '{loc_url}'
    WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
    """)
    print(f"Registered: {loc_name}  →  {loc_url}")


In [0]:
# ── Step 2: Validate External Locations ──────────────────────────────────────

for loc in ["bronze", "silver", "gold"]:
    loc_name = f"{STORAGE_ACCOUNT}_{loc}"
    df = spark.sql(f"DESCRIBE EXTERNAL LOCATION {loc_name}")
    print(f"\n── {loc_name} ──")
    df.display()


In [0]:
# ── Step 3: Smoke-test — read raw JSON directly from Bronze container ─────────
# Confirms the External Location and network routing are working end-to-end.
# Guarded with a try/except so a missing file doesn't block initial environment setup.

try:
    df = spark.read.option("multiLine", "true").json(f"{BRONZE_PATH}drivers.json")
    print(f"drivers.json schema: {df.schema.simpleString()}")
    df.display()
except Exception as e:
    print(f"[WARN] Smoke-test skipped — file may not exist yet or path is unreachable.\n  {e}")
